# W10 Practice 1 — LangChain Core Patterns

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week10/W10_practice_langchain_core.ipynb)

**Goal.** Learn the standard way to use LangChain: call a model, get structured output, make tools, and build an agent that keeps a conversation.

**Cell labels.** The first line of each code cell says what to do with it:

- `# ▶ RUN · example`: read each line, then run.
- `# ▶ RUN · setup`: just run.
- `# ✍️ TRY · exercise`: write your code in the cell, then run (Part 5).

It takes about 90 minutes and is not collected.

## LangChain

An application that uses a language model needs the same parts again and again: calls to the model, messages, tools, and the loop of an agent. Written by hand with the SDK of one provider, this code works only with that provider.

**LangChain** is an open-source framework that provides these parts as ready-made components. Its main features:

1. **One interface for many providers.** Models from OpenAI, Anthropic, Google, or a local server are made and called in the same way.
2. **Standard components.** Messages, tools, and structured output work the same way for every provider.
3. **Integrations.** Text splitters, embedding models, and vector stores for RAG come ready-made.
4. **Ready-made agents.** One function builds an agent. It runs on LangGraph, a library from the same developers that runs an agent as a graph of steps.

| Part | What you learn | Main names |
|---|---|---|
| 1. Invocation | make a model and call it with a question or a list of messages | `init_chat_model`, `invoke` |
| 2. Structured output | get fixed fields instead of text | a Pydantic class, `with_structured_output` |
| 3. Tools | turn a function into a tool | `@tool` |
| 4. The agent | build an agent and continue a conversation in a thread | `create_agent`, `InMemorySaver` |
| 5. Exercises | use Parts 1 to 4 on a small shop | |

## 0. Setup

*Do:* run the two cells; paste your API key into the second.

In [ ]:
# ▶ RUN · setup: just run
%pip install -q "langchain==1.4.4" "langchain-openai==1.7.0" "langgraph==1.2.14"

In [ ]:
# ▶ RUN · setup: just run
import os

OPENAI_API_KEY = "PASTE-YOUR-KEY-HERE"   # paste your key between the quotes
if OPENAI_API_KEY != "PASTE-YOUR-KEY-HERE":
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY   # LangChain reads the key from here

## 1. Invocation

A program uses a language model by sending it input and reading the reply. In LangChain this is one method, `invoke`, on a model object. The input is a string or a list of messages, and the reply is a message from the model.

You make the model object once with `init_chat_model`, naming the provider and the model, and then call `invoke` on it as often as you need.

### 1.1 The model object

*Do:* run the cell. It makes the model object that the whole notebook uses.

In [ ]:
# ▶ RUN · example: read each line, then run
from langchain.chat_models import init_chat_model

MODEL = "openai:gpt-4.1"                           # "provider:model": the part before the colon names the provider
model = init_chat_model(MODEL)                     # for another provider, change only MODEL

### 1.2 A question

*Do:* run the cell. It sends one question as a string and prints the text of the reply and the number of tokens that the call used.

In [ ]:
# ▶ RUN · example: read each line, then run
QUESTION = "In one sentence, what is a canceled order in an online shop?"
reply = model.invoke(QUESTION)                     # a string is sent as one user message; the reply is an AIMessage
print(reply.content)                               # the text of the reply
usage = reply.usage_metadata                       # a dictionary of token counts; a call is charged per token
print("input tokens:", usage["input_tokens"], "| output tokens:", usage["output_tokens"])

### 1.3 A list of messages

A string becomes one user message. To give the model instructions as well, you pass a list of messages. Each message is an object, and its class names its role:

| Class | Role |
|---|---|
| `SystemMessage` | instructions for the model |
| `HumanMessage` | a message from the user |
| `AIMessage` | a reply of the model |
| `ToolMessage` | the result of a tool, added by the agent (Part 4) |

Every LangChain message has the method `pretty_print`, which prints the message in a readable form: a header with its type, then its content. Use it to check what a list of messages holds; this notebook checks every message with it from here on.

*Do:* run the cell. It calls the model with a system message and a user message, then prints the two messages and the reply.

In [ ]:
# ▶ RUN · example: read each line, then run
from langchain.messages import SystemMessage, HumanMessage

STAFF_SYSTEM = "You answer questions from shop staff. Answer in at most two sentences."
messages = [
    SystemMessage(STAFF_SYSTEM),                   # instructions for the model
    HumanMessage("A customer ordered 2 mugs and 1 desk lamp. List the items in the order."),
]
reply = model.invoke(messages)
for m in messages:
    m.pretty_print()                               # a header with the type of the message, then its content
reply.pretty_print()

**Reading the output.** The reply has at most two sentences, as the system message asks. The model keeps nothing between calls; keeping a conversation is the job of the agent in Part 4.

## 2. Structured Output

Many programs need values from the model's answer, not only text to show. A shop program that reads customer messages, for example, needs each product, its quantity, and the delivery city as separate values. Free text does not give these reliably, because the same order can be written in many ways. **Structured output** solves this: the model returns its answer in fixed fields, so your code can read each value directly.

In LangChain this takes two steps:

1. Describe the result as a **Pydantic** class. Pydantic is a Python library, installed with LangChain, for classes whose fields have fixed types. Each field of the class becomes a value that the model must fill in.
2. Give the class to the model with `with_structured_output`. The new model returns an object of that class instead of text.

This method takes one class. A more general method, in which the model chooses among several classes, is introduced later.

### 2.1 An order as an object

*Do:* run the cell. It extracts an order from a customer message and prints the fields of the result.

In [ ]:
# ▶ RUN · example: read each line, then run
from pydantic import BaseModel

## Step 1 - describe the result as Pydantic classes
class Item(BaseModel):                             # inheriting from BaseModel makes a Pydantic class
    product: str                                   # one field per line: name, then type
    quantity: int                                  # Pydantic checks that the value is a whole number

class Order(BaseModel):                            # one schema: Order holds Item, so the model fills a single Order
    items: list[Item]                              # a list of Item objects
    city: str

## Step 2 - give the class to the model
order_model = model.with_structured_output(Order)  # sends the class as a JSON schema; replies come back as Order objects

customer_message = "Hi, I'd like to order 3 notebooks and 2 pen sets, delivered to Busan."
# the prompt says only what to extract; the fields come from the class
order = order_model.invoke("Extract the order from the customer message.\n\nCustomer message: " + customer_message)
print(order)                                       # an Order object, not text
for item in order.items:                           # read the fields with a dot
    print(item.product, item.quantity)
print(order.city)

**Reading the output.** The result is an Order object, not text. Its fields hold the values that the model extracted, with the types that the class declares.

## 3. Tools

A model writes text from what it learned in training. It cannot look up the current weather or check a calculation. A **tool** is a function that your code runs for the model: the model requests the tool by name with arguments, the function runs, and the result goes back to the model in a message.

In LangChain you make a tool by writing `@tool` on the line above a Python function. LangChain then builds what the model reads about the tool: the function name becomes the tool name, the docstring becomes its description, and the type hints give the type of each argument. The model chooses a tool from the name and the description alone, so the docstring must state clearly what the tool does and what each argument means.

Running a tool when the model requests it is the job of the agent in Part 4. Here you only make the tools and check them.

### 3.1 A weather tool

*Do:* run the cell. It makes a weather tool, prints its name, description, and arguments, and runs the tool once by hand.

In [ ]:
# ▶ RUN · example: read each line, then run
import json
import requests
from langchain.tools import tool

@tool                                              # makes get_weather a tool object
def get_weather(city: str) -> str:
    """Get the current temperature of a city in Celsius.

    Args:
        city: City name in English, for example "Seoul".
    """
    try:                                           # on a network or lookup error, return the error as text
        place = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                             params={"name": city, "count": 1}, timeout=10).json()["results"][0]   # first match
        weather = requests.get("https://api.open-meteo.com/v1/forecast",
                               params={"latitude": place["latitude"], "longitude": place["longitude"],
                                       "current": "temperature_2m"}, timeout=10).json()
        return json.dumps({"city": place["name"], "temperature_c": weather["current"]["temperature_2m"]})   # text: it goes back in a message
    except Exception as error:
        return "error: " + str(error)

print(get_weather.name)                            # the tool name: the function name
print(get_weather.description)                     # the description: the docstring
print(get_weather.args)                            # the arguments and their types: from the type hints
print(get_weather.invoke({"city": "Seoul"}))       # run the tool by hand: the arguments as one dictionary

**Reading the output.** The name, the description, and the arguments are what the model reads about the tool. If the weather service cannot be reached, the tool returns the error as text, and you can continue.

### 3.2 A calculator tool

*Do:* run the cell. It makes a second tool, so that one question can need two tools in a row (Part 4).

In [ ]:
# ▶ RUN · example: read each line, then run
@tool
def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression and return the result.

    Args:
        expression: A Python arithmetic expression, for example "58 * 94 + 37 * 21".
    """
    try:
        return str(eval(expression, {"__builtins__": {}}))   # run the text as Python, without built-in functions
                                                             # (enough for this practice, not a real safeguard)
    except Exception as error:
        return "error: " + str(error)

print(calculate.invoke({"expression": "58 * 94 + 37 * 21"}))

## 4. The Agent

A question can need several tools one after another, for example the temperature first and then a calculation with it, and the number of steps is not known in advance. An **agent** handles this with a loop. It calls the model with the tools; if the reply requests tools, it runs them, adds their results to the message list, and calls the model again; if the reply has no requests, that reply is the answer.

LangChain provides this loop ready-made. You create the agent with `create_agent`, giving the model, the list of tools, and the system prompt, and then call `invoke` with your question. The input and the result of `invoke` are a **state**, a dictionary in which the agent keeps its data. Its main entry is the message list under the key `"messages"`; the state that comes back holds the question, every request, every tool result, and the answer.

### 4.1 A question that needs two tools

*Do:* run the cell. It asks for the current temperature in Seoul in Fahrenheit, which needs the weather tool and then the calculator.

In [ ]:
# ▶ RUN · example: read each line, then run
from langchain.agents import create_agent

AGENT_SYSTEM = "You answer questions with your tools. Use calculate for every calculation."
QUESTION = "What is the current temperature in Seoul in Fahrenheit? Use F = C × 1.8 + 32."

agent = create_agent(model, tools=[get_weather, calculate], system_prompt=AGENT_SYSTEM)
state = agent.invoke({"messages": [HumanMessage(QUESTION)]})   # the input state: the message list under "messages"
for m in state["messages"]:                                    # the state that comes back holds the whole run
    m.pretty_print()

**Reading the output.** The list shows the loop. In our runs, the model requested the weather, the agent ran the tool and added its result, the model requested the calculation with that result, and then it answered. Each request is followed by a tool message with its result. The system prompt is not in the list, because the agent adds it to each model call without storing it.

### 4.2 A conversation in a thread

A state lasts for one call, so by default the agent does not remember the previous question. To continue a conversation, you give the agent a **checkpointer**, which saves the state of each conversation under a **thread**. You pass the checkpointer once, when you create the agent, and the thread id with each call. In the same thread you send only the new question: the agent starts from the saved messages, including the earlier tool calls and their results. This is the **short-term memory** of the agent.

The thread id goes in the **config**, the second argument of `invoke`. The config holds the settings of a run, apart from its input. Settings for the whole run are top-level keys, for example `recursion_limit`, the maximum number of steps. Values that parts of the agent read during the run go under `"configurable"`; the checkpointer reads the `thread_id` there.

*Do:* run the cell. It asks the question of 4.1 in thread "1", then the follow-up "What about Busan?" in the same thread, and prints the messages of the thread.

In [ ]:
# ▶ RUN · example: read each line, then run
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(model, tools=[get_weather, calculate], system_prompt=AGENT_SYSTEM,
                            checkpointer=InMemorySaver())   # saves the state of each thread in memory
config = {"configurable": {"thread_id": "1"}}               # the config of a run: the checkpointer reads thread_id under "configurable"

## First question: the thread starts
memory_agent.invoke({"messages": [HumanMessage(QUESTION)]}, config=config)

## Second question: only the new question, in the same thread
state = memory_agent.invoke({"messages": [HumanMessage("What about Busan?")]}, config=config)
for m in state["messages"]:                                  # the state of the thread: both questions with their tool calls
    m.pretty_print()

**Reading the output.** The list starts with the first question, its tool calls, and its answer, which the thread kept. The follow-up names only the city; the agent took the rest of the question, the temperature in Fahrenheit, from the thread and called both tools again for Busan. In our runs, each question left six messages in the thread: the question, two pairs of a request and its result, and the answer, so the thread held 12.

## 5. Exercises

The exercises use what you learned in Parts 1 to 4 on a small shop. Each task says what to write and what to expect. Write the code in the cell under the task, run it, and compare your output with the expected result. The exercises are not collected.

### 5.1 A system message

Call the model with the system message and the question below, as in 1.3. Print the text of the reply and the total number of tokens of the call (the key `total_tokens`).

Expected: one word, "Yes".

In [ ]:
# ✍️ TRY · exercise: write your code in this cell, then run
ONE_WORD_SYSTEM = "Answer with one word."
MUG_QUESTION = "Is a mug used for drinking?"

# 1. the list of messages: the system message and the question
# 2. call the model, then print the text of the reply and its total number of tokens

### 5.2 A return request

Write a Pydantic class `ReturnRequest` with three fields: `product` (text), `reason` (text), and `wants_refund` (true or false; the type is `bool`). Extract a `ReturnRequest` from the message below and print it.

Expected: the product is the desk lamp, and `wants_refund` is `True`.

In [ ]:
# ✍️ TRY · exercise: write your code in this cell, then run
RETURN_MESSAGE = "The desk lamp I got yesterday flickers all the time. Please take it back and give me my money back."

# 1. the class ReturnRequest with the fields product, reason, and wants_refund
# 2. a model that returns ReturnRequest objects
# 3. call it with an instruction and RETURN_MESSAGE, then print the result

### 5.3 A price tool

The function below looks up a price in the shop. Turn it into a tool: write `@tool` above it and a docstring that says what the tool does and what `product` must be. The model sees only the name and the docstring, so say that the product name is singular and lower case, and list the four products. Print the description of the tool, then run the tool once for `"mug"`.

Expected: `{"product": "mug", "price": 8}`.

In [ ]:
# ✍️ TRY · exercise: write your code in this cell, then run
PRICES = {"mug": 8, "desk lamp": 35, "notebook": 3, "pen set": 6}   # prices in dollars

# add the decorator and the docstring
def get_price(product: str) -> str:
    if product in PRICES:
        return json.dumps({"product": product, "price": PRICES[product]})
    return "error: unknown product " + product

# print the description, then run the tool for "mug"

### 5.4 A shop agent

Make an agent with the tools `get_price` and `calculate` (3.2), the system message below, and a checkpointer. Ask the question below in the thread `SHOP_THREAD` and print the messages of the result.

Expected: the agent requests the prices, then a calculation, and answers 51 dollars.

In [ ]:
# ✍️ TRY · exercise: write your code in this cell, then run
SHOP_SYSTEM = "You answer questions about prices in the shop. Use get_price for every price and calculate for every calculation."
SHOP_QUESTION = "How much do 2 mugs and 1 desk lamp cost in total?"
SHOP_THREAD = {"configurable": {"thread_id": "shop-1"}}

# 1. the agent: the model, the tools get_price and calculate, SHOP_SYSTEM, and a checkpointer
# 2. invoke it with SHOP_QUESTION in SHOP_THREAD and print the messages of the result

### 5.5 A follow-up question

Continue the conversation of 5.4: send only the question below in the same thread, and print the last message of the result.

Expected: 16 dollars, because only the two mugs are left.

In [ ]:
# ✍️ TRY · exercise: write your code in this cell, then run
FOLLOW_UP_SHOP = "The lamp was canceled. What is the new total?"

# invoke the agent of 5.4 with FOLLOW_UP_SHOP in SHOP_THREAD and print the last message